# Task 2: Biomaker Embedding

This task is about giving biological meaning to the biomarkers (glycans) we previously discovered by embedding them (i.e. converting each glycan into a numerical representation) into a space that reflects their biochemical, functional, and clinical relationships (i.e. such that similar glycans are placed close together in the embedding space based on these features).

In this part, we will:
- Learn a meaningful embedding space that captures relationships between glycans based on structure, origin, tissue, disease, and protein interactions.
- Validate the embedding by using the N-glycans (known structures) as a ground-truth set
- Embed our discovered glycans into this space and assess their closeness to other structures to draw conclusions as to their nature.

In [95]:
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
import time

# Machine Learning methods
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA

# Glycowork
from glycowork.motif.processing import min_process_glycans

In [ ]:
# Load the data
glycan_list = pd.read_csv("./data/glycan_embedding/glycan_list.csv")
df_glycan = pd.read_pickle("./data/glycan_embedding/df_glycan.pkl")
glycan_binding = pd.read_pickle("./data/glycan_embedding/glycan_binding.pkl")
N_glycans_df = pd.read_pickle("./data/glycan_embedding/N_glycans_df.pkl")

# Print the shape of each dataset
print(f"Shape of Glycan List (our discovered molecules): {glycan_list.shape}")
print(f"Shape of Glycan Dataset (sequences, species, tissue, disease): {df_glycan.shape}")
print(f"Shape of Protein-Glycan binding interactions Dataset: {glycan_binding.shape}")
print(f"Shape of N-glycans Dataset (N-gylcans sequences for control representation): {N_glycans_df.shape}")

Shape of Glycan List (our discovered molecules): (5, 4)
Shape of Glycan Dataset (sequences, species, tissue, disease): (50589, 23)
Shape of Protein-Glycan binding interactions Dataset: (1465, 2745)
Shape of N-glycans Dataset (N-gylcans sequences for control representation): (48, 24)


In [45]:
# Missing Values
print(f"Glycan List:\n{glycan_list.isna().sum()}\n")
print(f"Glycan DF:\n{df_glycan.isna().sum()}\n")
print(f"Glycan Binding:\n{glycan_binding.isna().sum()}\n")
print(f"N-Glycans:\n{N_glycans_df.isna().sum()}")

Glycan List:
glycan                0
Composition           0
tissue_species        0
tissue_sample         0
Processed Sequence    0
dtype: int64

Glycan DF:
glycan                     0
Species                    0
Genus                      0
Family                     0
Order                      0
Class                      0
Phylum                     0
Kingdom                    0
Domain                     0
ref                        0
glytoucan_id           15573
glycan_type            23628
disease_association        0
disease_id                 0
disease_sample             0
disease_direction          0
disease_ref                0
disease_species            0
tissue_sample              0
tissue_id                  0
tissue_ref                 0
tissue_species             0
Composition                0
dtype: int64

Glycan Binding:
3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S                                                                  1461
3-Anhydro-Gal(a1-3)Gal

## Part 1: Learn a Glycan Embedding Space

In part 1 of our analysis, we will build a feature space that places glycans near each other if they:
- Have similar sequences or compositions
- Come from similar species or tissues
- Are involved in the same diseases
- Bind to the same proteins

To do this, we will use machine learning to learn and validate the embedding, starting with simpler, interpretable models and eventually scaling-up to more complex architectures.

### Part 1A: Build Feature Matrix for each Glycan

For all gylcans in `df_glycan` and `glycan_list`, we will extract the following features:
- Sequence Proximity: we will investigate ways to define proximity between two sequences of glycans.
- Composition
- Species and Tissue
- Disease Association
- Protein-Glycan Binding

In [19]:
# Load df_glycan as is done in glycowork
df_glycan2 = copy.deepcopy(df_glycan)
df_glycan2.set_index("glycan", inplace = True)
df_glycan2.head(1).style.set_properties(**{'font-size': '8pt', 'font-family': 'Helvetica','border-collapse': 'collapse','border': '1px solid black'})

For glycan sequence embeddings, we will treat each glycan sequence as a "sentence" of glycoletters. We will try two embedding techniques:
- TF-IDF vectorizer: this converts a collection of text to a matrix of TF-IDF features. 
- Count vectorizer: this converts a collection of text to a matrix of token counts. 

Later on, once we have developed various embedding techniques for the rest of the features, we will try all combinations of embedding techniques to see which one is the best.

Mabye add PCA?

In [ ]:
# Glycan Sequence Proximity
# Use glycowork's min_process_glycans() to convert glycans sequence into a nested lists of glycoletters
df_glycan2['Processed Sequence'] = min_process_glycans(list(df_glycan2.index))

In [139]:
# Join glycoletters into space-separated "sentences"
df_glycan2['Sequence_str'] = df_glycan2['Processed Sequence'].apply(lambda seq: ' '.join(seq))

# Fit TF-IDF vectorizer on glycoletter sentences
# This turns each glycan into a fixed-length vector of TF-IDF scores
vectorier = TfidfVectorizer()
t0 = time.time()
X_tfidf = vectorier.fit_transform(df_glycan2['Sequence_str'])
print(f"TF-IDF Vectorizer time: {round(time.time() - t0, 2)} seconds")
# Save TF-IDF glycan embeddings based on sequence similarity
embeddings_tfidf = X_tfidf.toarray()
# Convert TF-IDF matrix to DataFrame
emb_seq_tfidf = pd.DataFrame(
    embeddings_tfidf,
    index=df_glycan2.index,  # assumes index contains glycan identifiers
    columns=vectorier.get_feature_names_out()  # glycoletter features
)

# PCA
pca = PCA(n_components=100)
t0 = time.time()
reduced_tfidf = pca.fit_transform(embeddings_tfidf)
print(f"PCA TF-IDF time: {round(time.time() - t0, 2)} seconds")
# Convert PCA-reduced embeddings to DataFrame
emb_seq_tfidf_pca = pd.DataFrame(
    reduced_tfidf,
    index=df_glycan2.index,  # use glycan identifiers
    columns=[f"PC{i+1}" for i in range(reduced_tfidf.shape[1])]  # name components
)


# Fit Counts vectorizer on glycoletter sentences
# This turns each gylcan into a fixed-length vector of token counts
vec = CountVectorizer(token_pattern=r"[^ ]+")
t0 = time.time()
X_counts = vec.fit_transform(df_glycan2['Sequence_str'])
print(f"Count Vectorizer time: {round(time.time() - t0, 2)} seconds")
# Save Counts glycan embeddings based on sequence similarity
embeddings_counts = X_counts.toarray()
# Create DataFrame with glycoletter features
emb_seq_counts = pd.DataFrame(
    embeddings_counts,
    index=df_glycan2.index,
    columns=vec.get_feature_names_out()
)

# PCA
pca = PCA(n_components=100)
t0 = time.time()
reduced_counts = pca.fit_transform(embeddings_counts)
print(f"PCA Counts time: {round(time.time() - t0, 2)} seconds")
# Convert PCA-reduced embeddings to DataFrame
emb_seq_counts_pca = pd.DataFrame(
    reduced_counts,
    index=df_glycan2.index,  # use glycan identifiers
    columns=[f"PC{i+1}" for i in range(reduced_counts.shape[1])]  # name components
)

TF-IDF Vectorizer time: 0.25 seconds
PCA TF-IDF time: 4.2 seconds
Count Vectorizer time: 0.28 seconds
PCA Counts time: 5.71 seconds


Glycan composition refers to monosaccharide types and their counts. We want to create an embedding space where glycans with similar compositions (similar sugar types and counts) are close together. We will use counts of monosaccharides as the embedding.

Could also use composition_to_mass or glycan_to_mass

In [140]:
# Get all unique monosaccharies across the dataset
monosaccharides = set()
for composition in df_glycan2['Composition']:
    monosaccharides.update(composition.keys())

# Create a DataFrame with monosaccharide counts
emb_composition = pd.DataFrame([
    {mono: composition.get(mono, 0) for mono in monosaccharides}
    for composition in df_glycan2['Composition']
])

In [141]:
# To use later for glycan list composition embedding!!
"""
import ast

# Convert composition strings into dictionaries
glycan_list['Composition_dict'] = glycan_list['Composition'].apply(ast.literal_eval)

# Get all unique monosaccharies across the dataset
monosaccharides = set()
for composition in glycan_list['Composition_dict']:
    monosaccharides.update(composition.keys())

# Create a DataFrame with monosaccharide counts
composition_df = pd.DataFrame([
    {mono: composition.get(mono, 0) for mono in monosaccharides}
    for composition in glycan_list['Composition_dict']
])
"""

"\nimport ast\n\n# Convert composition strings into dictionaries\nglycan_list['Composition_dict'] = glycan_list['Composition'].apply(ast.literal_eval)\n\n# Get all unique monosaccharies across the dataset\nmonosaccharides = set()\nfor composition in glycan_list['Composition_dict']:\n    monosaccharides.update(composition.keys())\n\n# Create a DataFrame with monosaccharide counts\ncomposition_df = pd.DataFrame([\n    {mono: composition.get(mono, 0) for mono in monosaccharides}\n    for composition in glycan_list['Composition_dict']\n])\n"

In [143]:
glycan_list

,glycan,Composition,tissue_species,tissue_sample,Processed Sequence,Composition_dict
0,Fuc(a1-?)GlcNAc(b1-2)Man(a1-6)[GlcNAc(b1-2)Man...,"{'dHex': 2, 'HexNAc': 4, 'Hex': 3}",['Homo_sapiens'],['blood'],"[Fuc, a1-?, GlcNAc, b1-2, Man, a1-6, GlcNAc, b...","{'dHex': 2, 'HexNAc': 4, 'Hex': 3}"
1,Neu5Ac(a2-?)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Glc...,"{'Neu5Ac': 1, 'Hex': 4, 'HexNAc': 4, 'dHex': 1}",['Homo_sapiens'],['blood'],"[Neu5Ac, a2-?, Gal, b1-4, GlcNAc, b1-2, Man, a...","{'Neu5Ac': 1, 'Hex': 4, 'HexNAc': 4, 'dHex': 1}"
2,Neu5Ac(a2-6)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Gal...,"{'Neu5Ac': 1, 'Hex': 5, 'HexNAc': 4}",['Homo_sapiens'],['blood'],"[Neu5Ac, a2-6, Gal, b1-4, GlcNAc, b1-2, Man, a...","{'Neu5Ac': 1, 'Hex': 5, 'HexNAc': 4}"
3,Neu5Ac(a2-6)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Glc...,"{'Neu5Ac': 1, 'Hex': 4, 'HexNAc': 4}",['Homo_sapiens'],['blood'],"[Neu5Ac, a2-6, Gal, b1-4, GlcNAc, b1-2, Man, a...","{'Neu5Ac': 1, 'Hex': 4, 'HexNAc': 4}"
4,Fuc(a1-2)[GalNAc(a1-3)]Gal(b1-4)GlcNAc(b1-2)Ma...,"{'dHex': 1, 'HexNAc': 5, 'Hex': 5}",['Homo_sapiens'],['blood'],"[Fuc, a1-2, GalNAc, a1-3, Gal, b1-4, GlcNAc, b...","{'dHex': 1, 'HexNAc': 5, 'Hex': 5}"


In [153]:
print(df_glycan2['Species'][0])
print(df_glycan2['tissue_species'][0])
print(df_glycan2['Genus'][0])
print(df_glycan2['Order'][0])
print(df_glycan2['Class'][0])
print(df_glycan2['Phylum'][0])
print(df_glycan2['Kingdom'][0])
print(df_glycan2['Domain'][0])

['Acinonyx_jubatus', 'Addax_nasomaculatus', 'Aepyceros_melampus', 'Ailuropoda_melanoleuca', 'Alcelaphus_buselaphus', 'Alces_alces', 'Alces_canadensis', 'Allochrocebus_lhoesti', 'Alouatta_palliata', 'Antidorcas_marsupialis', 'Antilocapra_americana', 'Balaena_mysticetus', 'Balaenoptera_acutorostrata', 'Balaenoptera_borealis', 'Balaenoptera_edeni', 'Bettongia_gaimardi', 'Bison_bison', 'Bos_frontalis', 'Bos_grunniens', 'Bos_indicus', 'Bos_taurus', 'Bubalus_arnee', 'Bubalus_bubalis', 'Callithrix_jacchus', 'Callorhinus_ursinus', 'Camelus_bactrianus', 'Camelus_dromedarius', 'Canis_familiaris', 'Canis_lupus', 'Capra_aegagrus_hircus', 'Capra_hircus', 'Castor_fiber', 'Cavia_porcellus', 'Ceratotherium_simum', 'Cervus_elaphus', 'Cervus_nippon', 'Chinchilla_chinchilla', 'Chlorocebus_pygerythrus', 'Choeropsis_liberiensis', 'Connochaetes_gnou', 'Connochaetes_taurinus', 'Crocuta_crocuta', 'Cystophora_cristata', 'Damaliscus_lunatus', 'Damaliscus_pygargus', 'Dasypus_novemcinctus', 'Dasyurus_maculatus', 

/var/folders/f6/p2jq7zvd7jq9y5mvqb6p1wlh0000gn/T/ipykernel_2448/34950465.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(df_glycan2['Species'][0])
/var/folders/f6/p2jq7zvd7jq9y5mvqb6p1wlh0000gn/T/ipykernel_2448/34950465.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(df_glycan2['tissue_species'][0])
/var/folders/f6/p2jq7zvd7jq9y5mvqb6p1wlh0000gn/T/ipykernel_2448/34950465.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo